# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmed-khaled123/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** one row = one page (`client_hash_id` + `content_hash_id`) for the month of
**March 2026** — a mid-panel month, per the `flyrank-data` warning to never develop label logic on
the `_sample` table (that is the sealed final month, June 2026).

**Time window, split in two:**
- **Prior window:** March 1-15 -- every feature below is built ONLY from these 15 days.
- **Target window:** March 16-31 -- used only to compute the label, never as a feature.

This is a small-scale version of the lane guide's "prior window -> future window" pattern, done at
month-scale because I currently have one downloaded partition (`month=2026-03`) rather than two
full months. Section 4 names this as an explicit limitation.

In [1]:
import os
import duckdb, pandas as pd

candidates = [
    os.path.expanduser("~/Documents/flyrank-hf-data"),      # your real Mac path
    os.path.expanduser("~/mnt/Documents/flyrank-hf-data"),  # Cowork sandbox bridge path
]
BASE = next((p for p in candidates if os.path.isdir(p)), candidates[0])
print("Using local warehouse folder:", BASE)

con = duckdb.connect()
FACT = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')"
DIM_CONTENT = f"read_parquet('{BASE}/dim_content.parquet')"

print(con.sql(f"DESCRIBE SELECT * FROM {FACT}").df().to_string())


Using local warehouse folder: /sessions/rcw-01sg1afdfkqjrxmdmesxl4zj/mnt/Documents/flyrank-hf-data
                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  Y

## 2. Fields: feature / label / context / excluded

**Tables used:** `fact_content_daily_performance` (`month=2026-03` partition only) is the primary
source; `dim_content` is joined in for two static fields.

- **Feature** (all from the PRIOR window, March 1-15, so they are knowable before the target
  window even starts): `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (aggregated per page),
  plus `word_count` and a derived `content_age_days` from `dim_content.content_created_date`
  (static content metadata that existed before March even began).
- **Label / proxy:** `is_declining` = 1 if the page's average daily clicks in the TARGET window
  (Mar 16-31) are lower than its average daily clicks in the PRIOR window (Mar 1-15). This is a
  proxy, not a confirmed multi-month outcome -- named again in Section 4.
- **Context (grouping/joins only, never features):** `client_hash_id`, `content_hash_id`.
- **Excluded, with a why:** `clicks_target` and `impressions_target` (the target-window raw
  counts) are excluded from the feature set on purpose -- they are literally the ingredients the
  label is computed from, so using them as features would just teach a model to read the label off
  itself. Section 3's trap proves this out loud.

In [2]:
sample = con.sql(f"""
    SELECT content_hash_id, word_count, content_created_date
    FROM {DIM_CONTENT}
    LIMIT 5
""").df()
sample


,content_hash_id,word_count,content_created_date
0,content_004de9653278b5a4,2555,2026-05-30
1,content_00dc5efae381b2ab,2430,2026-06-12
2,content_01410f2556c327ac,2645,2026-05-09
3,content_019f27f634053ca7,2522,2026-06-15
4,content_01efa71faea45dcc,2552,2026-05-21


## 3. Verify it with queries (grain, counts, availability) + five features + the trap

Three small queries on the March 2026 partition, then a five-feature frame built only from the
prior window, then the deliberate leak.

### Query 1 -- grain: is one row really one (day, client, content)?

In [3]:
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"duplicate (report_date, client_hash_id, content_hash_id) rows found: {len(dupes)}")
print("Zero back confirms the grain: one row = one page on one day.")


duplicate (report_date, client_hash_id, content_hash_id) rows found: 0
Zero back confirms the grain: one row = one page on one day.


### Query 2 -- slice size and date span

In [4]:
span = con.sql(f"""
    SELECT COUNT(*) AS rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients, COUNT(DISTINCT content_hash_id) AS n_content
    FROM {FACT}
""").df()
span


,rows,min_date,max_date,n_clients,n_content
0,9841378,2026-03-01,2026-03-31,55,331437


### Query 3 -- availability, filtered with `IS TRUE`

In [5]:
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {FACT}
""").df()
avail["gsc_available_pct"] = 100 * avail["gsc_available_rows"] / avail["total_rows"]
avail["ga4_available_pct"] = 100 * avail["ga4_available_rows"] / avail["total_rows"]
print(avail.to_string(index=False))
print("\nOnly rows with gsc_data_available IS TRUE are real GSC observations -- the rest are")
print("zero-filled placeholders for clients/days without tracking yet. Everything below filters on this.")


 total_rows  gsc_available_rows  ga4_available_rows  gsc_available_pct  ga4_available_pct
    9841378           3611061.0            413966.0          36.692636           4.206382

Only rows with gsc_data_available IS TRUE are real GSC observations -- the rest are
zero-filled placeholders for clients/days without tracking yet. Everything below filters on this.


### Five features (built ONLY from the prior window, March 1-15)

Each feature gets one line: knowable at the decision moment because...

In [6]:
feat = con.sql(f"""
    WITH avail AS (SELECT * FROM {FACT} WHERE gsc_data_available IS TRUE),
    prior AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_prior,
               SUM(gsc_clicks)      AS clicks_prior,
               AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS avg_position_prior,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impr_prior
        FROM avail WHERE report_date <= DATE '2026-03-15'
        GROUP BY 1, 2
    ),
    target AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_target
        FROM avail WHERE report_date >= DATE '2026-03-16'
        GROUP BY 1, 2
    )
    SELECT p.*, COALESCE(t.clicks_target, 0) AS clicks_target
    FROM prior p LEFT JOIN target t USING (client_hash_id, content_hash_id)
    WHERE p.impressions_prior >= 10
""").df()

content = con.sql(f"SELECT content_hash_id, word_count, content_created_date FROM {DIM_CONTENT}").df()
feat = feat.merge(content, on="content_hash_id", how="left")
feat["content_age_days"] = (pd.Timestamp("2026-03-01") - pd.to_datetime(feat["content_created_date"])).dt.days
feat = feat[feat["content_age_days"] >= 0].copy()  # drop pages that did not exist yet at period start

feat["rate_prior"]  = feat["clicks_prior"]  / 15.0
feat["rate_target"] = feat["clicks_target"] / 16.0
feat["is_declining"] = (feat["rate_target"] < feat["rate_prior"]).astype(int)

print(f"{len(feat):,} pages with >=10 prior-window impressions; decline rate: {feat['is_declining'].mean():.3f}\n")

print("1. impressions_prior      -- knowable because it is already logged by GSC on day 15, before the target window opens.")
print("2. clicks_prior           -- same: an observed count from days already past by the decision moment.")
print("3. avg_position_prior     -- same: the page's average rank over days 1-15 only.")
print("4. days_with_impr_prior   -- same: how many of the first 15 days the page showed up at all.")
print("5. content_age_days       -- static: content_created_date is always before March 1 by construction (filtered above).")

feat[["impressions_prior", "clicks_prior", "avg_position_prior", "days_with_impr_prior",
      "content_age_days", "is_declining"]].describe()


114,715 pages with >=10 prior-window impressions; decline rate: 0.298

1. impressions_prior      -- knowable because it is already logged by GSC on day 15, before the target window opens.
2. clicks_prior           -- same: an observed count from days already past by the decision moment.
3. avg_position_prior     -- same: the page's average rank over days 1-15 only.
4. days_with_impr_prior   -- same: how many of the first 15 days the page showed up at all.
5. content_age_days       -- static: content_created_date is always before March 1 by construction (filtered above).


,impressions_prior,clicks_prior,avg_position_prior,days_with_impr_prior,content_age_days,is_declining
count,114715.000000,114715.000000,114715.000000,114715.000000,114715.000000,114715.000000
mean,1094.048485,3.258127,15.764988,13.282910,174.753615,0.298339
std,3031.817332,15.129955,16.644371,2.830427,121.426835,0.457531
min,10.000000,0.000000,0.000000,1.000000,0.000000,0.000000
25%,56.000000,0.000000,4.958333,13.000000,53.000000,0.000000
50%,226.000000,0.000000,8.763709,15.000000,165.000000,0.000000
75%,892.000000,2.000000,20.595574,15.000000,244.000000,1.000000
max,161575.000000,2395.000000,127.620709,15.000000,464.000000,1.000000


### The trap: add one label-derived column on purpose

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["impressions_prior", "clicks_prior", "avg_position_prior",
                    "days_with_impr_prior", "content_age_days"]
model_df = feat.dropna(subset=honest_features).copy()
y = model_df["is_declining"]

Xtr, Xte, ytr, yte = train_test_split(model_df[honest_features], y, test_size=0.25,
                                       random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xtr, ytr)
honest_auc = roc_auc_score(yte, honest_model.predict_proba(Xte)[:, 1])
print(f"HONEST model (5 prior-window features only)          ROC-AUC: {honest_auc:.3f}")

# The trap: add clicks_target on purpose -- the exact column the label was built from.
leaky_features = honest_features + ["clicks_target"]
Xtrl, Xtel, ytrl, ytel = train_test_split(model_df[leaky_features], y, test_size=0.25,
                                           random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xtrl, ytrl)
leaky_auc = roc_auc_score(ytel, leaky_model.predict_proba(Xtel)[:, 1])
print(f"LEAKY model  (+ clicks_target, the label's own ingredient)  ROC-AUC: {leaky_auc:.3f}")

print(f"\nAdding clicks_target jumps ROC-AUC from {honest_auc:.3f} toward {leaky_auc:.3f} -- that jump IS the leak,")
print("not a real improvement: the model is just reading the label off a column the label was computed from.")
print("Deleting clicks_target and keeping the honest number is the right move -- done below.")

# Delete the leaky column and keep only the honest result.
del leaky_features, Xtrl, Xtel, ytrl, ytel, leaky_model, leaky_auc
print(f"\nFinal, honest, kept number -- Precision-relevant metric ROC-AUC: {honest_auc:.3f}")


HONEST model (5 prior-window features only)          ROC-AUC: 0.766


LEAKY model  (+ clicks_target, the label's own ingredient)  ROC-AUC: 1.000

Adding clicks_target jumps ROC-AUC from 0.766 toward 1.000 -- that jump IS the leak,
not a real improvement: the model is just reading the label off a column the label was computed from.
Deleting clicks_target and keeping the honest number is the right move -- done below.

Final, honest, kept number -- Precision-relevant metric ROC-AUC: 0.766


## 4. Data limits

**Named limitation: this is an intra-month proxy, not a true future-month outcome.** The label
compares the second half of March to the first half of the SAME month, because I have only
downloaded the `month=2026-03` partition so far. The lane guide's recommended design -- prior
90 days predicting the NEXT 30 days -- needs a real subsequent month (e.g. April 2026) that is
outside this slice; a 15/16-day intra-month split is a reasonable stand-in for a data contract
exercise, but it is not the same claim and should not be presented as one in later weeks.

A second, supporting limit shows up directly in Query 3: only ~37% of March rows have
`gsc_data_available IS TRUE` and ~4% have `ga4_data_available IS TRUE` -- most rows in any single
month are zero-filled placeholders for clients or days without tracking yet (the unbalanced panel
the `flyrank-data` skill warns about), not real zero-traffic observations.

In [8]:
print("gsc_data_available IS TRUE:", f"{avail['gsc_available_pct'][0]:.1f}%", "of March rows")
print("ga4_data_available IS TRUE:", f"{avail['ga4_available_pct'][0]:.1f}%", "of March rows")
print("Both numbers come straight out of Query 3 above -- reproduced here as the evidence for this limitation.")


gsc_data_available IS TRUE: 36.7% of March rows
ga4_data_available IS TRUE: 4.2% of March rows
Both numbers come straight out of Query 3 above -- reproduced here as the evidence for this limitation.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.